<a href="https://colab.research.google.com/github/Mr-Kondo/_Inbox/blob/main/test_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Google Colab RAGシステム - 完全版
# 性能比較機能付き

!python -m pip install -q docling sentence-transformers chromadb transformers accelerate bitsandbytes \
    torch pypdf pandas plotly langchain langchain-community rank-bm25

import os
import time
import json
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, asdict
import numpy as np

# Docling関連
from docling.document_converter import DocumentConverter

# Embedding & Vector DB
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# LLM
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Reranker
from sentence_transformers import CrossEncoder

# Chunking
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)

print("✅ ライブラリのインポート完了")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ================================================================================
# 設定クラス
# ================================================================================

@dataclass
class ChunkingConfig:
    """チャンキング設定"""
    method: str  # "recursive" or "semantic"
    chunk_size: int
    chunk_overlap: int
    overlap_ratio: float  # 重複率（0.0-1.0）

    def __post_init__(self):
        # overlap_ratioからchunk_overlapを計算
        if self.overlap_ratio > 0:
            self.chunk_overlap = int(self.chunk_size * self.overlap_ratio)

@dataclass
class RAGConfig:
    """RAG全体設定"""
    use_reranker: bool
    chunking: ChunkingConfig
    top_k: int = 5
    rerank_top_k: int = 3

@dataclass
class ExperimentResult:
    """実験結果"""
    config: RAGConfig
    query: str
    answer: str
    execution_time: float
    retrieved_chunks: List[str]
    scores: List[float]


In [3]:
# ================================================================================
# PDFドキュメント処理
# ================================================================================

class DocumentProcessor:
    """Doclingを使用したPDF処理"""

    def __init__(self):
        self.converter = DocumentConverter()

    def process_pdf(self, pdf_path: str) -> str:
        """PDFをテキストに変換"""
        print(f"📄 PDF処理開始: {pdf_path}")
        result = self.converter.convert(pdf_path)
        text = result.document.export_to_markdown()
        print(f"✅ 抽出完了: {len(text)}文字")
        return text

In [4]:
# ================================================================================
# チャンキング
# ================================================================================

class DocumentChunker:
    """複数のチャンキング手法を提供"""

    @staticmethod
    def chunk_recursive(text: str, config: ChunkingConfig) -> List[str]:
        """再帰的文字分割"""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
            separators=["\n\n", "\n", "。", "、", " ", ""]
        )
        chunks = splitter.split_text(text)
        print(f"🔪 再帰的チャンキング: {len(chunks)}チャンク生成")
        return chunks

    @staticmethod
    def chunk_semantic(text: str, config: ChunkingConfig,
                       embedder: SentenceTransformer) -> List[str]:
        """セマンティックチャンキング（意味的な区切り）"""
        # まず文単位で分割
        sentences = text.replace("。", "。\n").split("\n")
        sentences = [s.strip() for s in sentences if s.strip()]

        if len(sentences) <= 1:
            return sentences

        # 文埋め込みを計算
        embeddings = embedder.encode(sentences, show_progress_bar=False)

        # 隣接文間のコサイン類似度を計算
        similarities = []
        for i in range(len(embeddings) - 1):
            sim = np.dot(embeddings[i], embeddings[i+1]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1])
            )
            similarities.append(sim)

        # 類似度の低い箇所で分割（意味的な境界）
        threshold = np.percentile(similarities, 30)  # 下位30%を境界とする

        chunks = []
        current_chunk = sentences[0]
        current_length = len(sentences[0])

        for i, sentence in enumerate(sentences[1:], 1):
            # 意味的境界かつサイズ制約
            if (similarities[i-1] < threshold and
                current_length > config.chunk_size * 0.5):
                chunks.append(current_chunk)
                current_chunk = sentence
                current_length = len(sentence)
            else:
                current_chunk += sentence
                current_length += len(sentence)

                # 最大サイズに達したら強制分割
                if current_length > config.chunk_size:
                    chunks.append(current_chunk)
                    current_chunk = ""
                    current_length = 0

        if current_chunk:
            chunks.append(current_chunk)

        print(f"🧠 セマンティックチャンキング: {len(chunks)}チャンク生成")
        return chunks

    @classmethod
    def chunk(cls, text: str, config: ChunkingConfig,
              embedder: Optional[SentenceTransformer] = None) -> List[str]:
        """設定に応じてチャンキング"""
        if config.method == "semantic":
            if embedder is None:
                raise ValueError("セマンティックチャンキングにはembedderが必要です")
            return cls.chunk_semantic(text, config, embedder)
        else:
            return cls.chunk_recursive(text, config)

In [5]:
# ================================================================================
# ベクトルデータベース
# ================================================================================

class VectorDatabase:
    """ChromaDBを使用したベクトルストア"""

    def __init__(self, collection_name: str = "rag_collection"):
        self.client = chromadb.Client(Settings(anonymized_telemetry=False))
        self.collection_name = collection_name
        # 使用可能な公開モデルに変更
        self.embedder = SentenceTransformer('intfloat/multilingual-e5-large')
        print(f"🔮 intfloat/multilingual-e5-large 埋め込みモデル読み込み完了")

        # コレクション初期化
        try:
            self.client.delete_collection(name=collection_name)
        except:
            pass
        self.collection = self.client.create_collection(name=collection_name)

    def add_documents(self, chunks: List[str]):
        """ドキュメントをベクトルDBに追加"""
        print(f"💾 {len(chunks)}チャンクをベクトル化中...")

        embeddings = self.embedder.encode(
            chunks,
            show_progress_bar=True,
            batch_size=32
        )

        self.collection.add(
            embeddings=embeddings.tolist(),
            documents=chunks,
            ids=[f"chunk_{i}" for i in range(len(chunks))]
        )
        print("✅ ベクトルDB構築完了")

    def search(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        """クエリに基づいて検索"""
        query_embedding = self.embedder.encode([query])[0]

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        documents = results['documents'][0]
        distances = results['distances'][0]

        # 距離をスコアに変換（距離が小さいほどスコアが高い）
        scores = [1 / (1 + d) for d in distances]

        return documents, scores

In [6]:
# ================================================================================
# Reranker
# ================================================================================

class Reranker:
    """日本語対応Reranker"""

    def __init__(self):
        # 日本語対応のクロスエンコーダーモデル
        self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        print("🎯 Rerankerモデル読み込み完了")

    def rerank(self, query: str, documents: List[str],
               top_k: int = 3) -> Tuple[List[str], List[float]]:
        """ドキュメントを再ランキング"""
        pairs = [[query, doc] for doc in documents]
        scores = self.model.predict(pairs)

        # スコアでソート
        ranked_indices = np.argsort(scores)[::-1][:top_k]

        reranked_docs = [documents[i] for i in ranked_indices]
        reranked_scores = [float(scores[i]) for i in ranked_indices]

        return reranked_docs, reranked_scores

In [7]:
# ================================================================================
# LLM生成
# ================================================================================

class JapaneseLLM:
    """日本語特化LLM（ELYZA）"""

    def __init__(self, model_name: str = "elyza/ELYZA-japanese-Llama-2-7b"):
        print(f"🤖 LLM読み込み中: {model_name}")

        # 4bit量子化設定
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            # メモリ不足対策としてオフロードを有効化
            llm_int8_enable_fp32_cpu_offload=True,
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            # メモリ不足対策としてdevice_mapをautoに設定
            device_map="auto",
            trust_remote_code=True
        )
        print("✅ LLM読み込み完了")

    def generate(self, prompt: str, max_new_tokens: int = 512) -> str:
        """テキスト生成"""
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # プロンプト部分を除去
        response = response.split("回答：")[-1].strip()
        return response

In [8]:
# ================================================================================
# RAGシステム
# ================================================================================

class RAGSystem:
    """統合RAGシステム"""

    def __init__(self, config: RAGConfig):
        self.config = config
        self.vector_db = VectorDatabase()
        self.reranker = Reranker() if config.use_reranker else None
        self.llm = JapaneseLLM()

    def index_document(self, pdf_path: str):
        """ドキュメントをインデックス化"""
        # PDF処理
        processor = DocumentProcessor()
        text = processor.process_pdf(pdf_path)

        # チャンキング
        chunks = DocumentChunker.chunk(
            text,
            self.config.chunking,
            embedder=self.vector_db.embedder
        )

        # ベクトルDB構築
        self.vector_db.add_documents(chunks)

        return len(chunks)

    def query(self, question: str) -> Tuple[str, List[str], List[float], float]:
        """質問に回答"""
        start_time = time.time()

        # 1. ベクトル検索
        documents, scores = self.vector_db.search(question, top_k=self.config.top_k)

        # 2. Reranking（オプション）
        if self.reranker:
            documents, scores = self.reranker.rerank(
                question,
                documents,
                top_k=self.config.rerank_top_k
            )

        # 3. プロンプト構築
        context = "\n\n".join([f"[参考{i+1}]\n{doc}" for i, doc in enumerate(documents)])

        prompt = f"""以下の参考情報を基に、質問に回答してください。

参考情報：
{context}

質問：{question}

回答："""

        # 4. LLM生成
        answer = self.llm.generate(prompt)

        execution_time = time.time() - start_time

        return answer, documents, scores, execution_time

In [9]:
# ================================================================================
# 実験管理
# ================================================================================

class ExperimentManager:
    """複数設定での性能比較実験"""

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.results: List[ExperimentResult] = []

    def run_experiments(self, test_queries: List[str]):
        """実験実行"""

        # 実験設定の組み合わせ
        configs = [
            # 1. Rerankerなし、通常チャンキング、サイズ500、重複10%
            RAGConfig(
                use_reranker=False,
                chunking=ChunkingConfig("recursive", 500, 0, 0.1),
                top_k=5
            ),
            # 2. Rerankerあり、通常チャンキング、サイズ500、重複10%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("recursive", 500, 0, 0.1),
                top_k=5,
                rerank_top_k=3
            ),
            # 3. Rerankerあり、サイズ1000、重複20%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("recursive", 1000, 0, 0.2),
                top_k=5,
                rerank_top_k=3
            ),
            # 4. Rerankerあり、サイズ300、重複30%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("recursive", 300, 0, 0.3),
                top_k=5,
                rerank_top_k=3
            ),
            # 5. Rerankerあり、セマンティックチャンキング、サイズ500、重複10%
            RAGConfig(
                use_reranker=True,
                chunking=ChunkingConfig("semantic", 500, 0, 0.1),
                top_k=5,
                rerank_top_k=3
            ),
        ]

        for i, config in enumerate(configs, 1):
            print(f"\n{'='*70}")
            print(f"実験 {i}/{len(configs)}")
            print(f"Reranker: {config.use_reranker}")
            print(f"チャンキング: {config.chunking.method}")
            print(f"サイズ: {config.chunking.chunk_size}")
            print(f"重複率: {config.chunking.overlap_ratio*100:.0f}%")
            print(f"{'='*70}\n")

            # RAGシステム構築
            rag = RAGSystem(config)
            rag.index_document(self.pdf_path)

            # 各クエリで実験
            for query in test_queries:
                print(f"\n質問: {query}")
                answer, chunks, scores, exec_time = rag.query(query)

                result = ExperimentResult(
                    config=config,
                    query=query,
                    answer=answer,
                    execution_time=exec_time,
                    retrieved_chunks=chunks,
                    scores=scores
                )
                self.results.append(result)

                print(f"回答: {answer[:100]}...")
                print(f"実行時間: {exec_time:.2f}秒")

            # メモリ解放
            del rag
            torch.cuda.empty_cache()

    def generate_report(self) -> pd.DataFrame:
        """結果レポート生成"""
        data = []
        for r in self.results:
            data.append({
                'Reranker': 'あり' if r.config.use_reranker else 'なし',
                'チャンキング': r.config.chunking.method,
                'チャンクサイズ': r.config.chunking.chunk_size,
                '重複率': f"{r.config.chunking.overlap_ratio*100:.0f}%",
                '質問': r.query[:30] + '...',
                '実行時間(秒)': round(r.execution_time, 2),
                '平均スコア': round(np.mean(r.scores), 3)
            })

        df = pd.DataFrame(data)
        return df

    def visualize_results(self):
        """結果の可視化"""
        df = self.generate_report()

        # 実行時間の比較
        fig1 = go.Figure()
        for reranker in ['あり', 'なし']:
            subset = df[df['Reranker'] == reranker]
            fig1.add_trace(go.Bar(
                name=f"Reranker{reranker}",
                x=subset.index,
                y=subset['実行時間(秒)']
            ))
        fig1.update_layout(
            title="実行時間の比較",
            xaxis_title="実験番号",
            yaxis_title="実行時間（秒）",
            barmode='group'
        )
        fig1.show()

        # スコアの比較
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=df.index,
            y=df['平均スコア'],
            mode='lines+markers',
            name='平均検索スコア'
        ))
        fig2.update_layout(
            title="検索精度の比較",
            xaxis_title="実験番号",
            yaxis_title="平均スコア"
        )
        fig2.show()


In [10]:
# ================================================================================
# メイン実行
# ================================================================================

def main():
    """メイン処理"""

    # PDFファイルのアップロード
    from google.colab import files
    print("📁 PDFファイルをアップロードしてください")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]

    # テストクエリ
    test_queries = [
        "この文書の主要なテーマは何ですか？",
        "具体的な数値データを教えてください。",
        "結論や提言は何ですか？"
    ]

    # 実験実行
    manager = ExperimentManager(pdf_path)
    manager.run_experiments(test_queries)

    # 結果表示
    print("\n" + "="*70)
    print("📊 実験結果サマリー")
    print("="*70)
    df = manager.generate_report()
    print(df.to_string(index=False))

    # 可視化
    manager.visualize_results()

    print("\n✅ すべての実験が完了しました")

if __name__ == "__main__":
    main()

📁 PDFファイルをアップロードしてください


Saving 08-0471.pdf to 08-0471.pdf

実験 1/5
Reranker: False
チャンキング: recursive
サイズ: 500
重複率: 10%



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

[INFO] 2025-10-18 15:15:30,910 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-10-18 15:15:30,914 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-10-18 15:15:30,915 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-10-18 15:15:30,979 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-10-18 15:15:31,027 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2025-10-18 15:15:31,028 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx


✅ 抽出完了: 11425文字
🔪 再帰的チャンキング: 29チャンク生成
💾 29チャンクをベクトル化中...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ベクトルDB構築完了

質問: この文書の主要なテーマは何ですか？
回答: 文書の主要なテーマは、淀川の大型模型実験の計測方法とその結...
実行時間: 609.78秒

質問: 具体的な数値データを教えてください。
回答: 画像の枚数は30枚です。
画像間隔は1秒です。
相係数の關值は0.5です。
CaseAの長は2mです。
CaseBの長は4mです。
参考情報の表中の数値は、CaseAの長は2mです。
CaseBの長は...
実行時間: 421.50秒

質問: 結論や提言は何ですか？
回答: 質問1：淀川の流速分布測定とLSPIVの利用について 質問2：河川水管理の統合的情報化としての淀川水系LSPIVの利用について 質問3：淀川の流速分布測定とLSPIVの利用について
質問4：淀川の流速...
実行時間: 688.10秒

実験 3/5
Reranker: True
チャンキング: recursive
サイズ: 1000
重複率: 20%

🔮 intfloat/multilingual-e5-large 埋め込みモデル読み込み完了
🎯 Rerankerモデル読み込み完了
🤖 LLM読み込み中: elyza/ELYZA-japanese-Llama-2-7b


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 